In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.utils import check_array
from scipy.sparse import csr_matrix, hstack
from sklearn.metrics.pairwise import cosine_similarity
import joblib, os

In [ ]:
ARTICLES_PATH = "/kaggle/input/article/Article.csv"                    
CHECKPOINT_PATH = "/kaggle/input/news-validation-1000/checkpoint_1000.parquet"  

articles = pd.read_csv(ARTICLES_PATH)
parq = pd.read_parquet(CHECKPOINT_PATH)

In [ ]:
ARTICLES_PATH = "/kaggle/input/article/Article.csv"                    
CHECKPOINT_PATH = "/kaggle/input/news-validation-1000/checkpoint_1000.parquet"  

articles = pd.read_csv(ARTICLES_PATH)
parq = pd.read_parquet(CHECKPOINT_PATH)

for col in ["url", "title"]:
    if col in articles.columns:
        articles[col] = articles[col].astype(str).str.strip()
parq["url"] = parq["url"].astype(str).str.strip()
parq["original_title"] = parq["original_title"].astype(str).str.strip()

df = pd.merge(
    articles[["url","title","main_text"]].rename(columns={"title":"article_title","main_text":"article_text"}),
    parq[["url","original_title","true_news_title","true_news_body","fake_news_title","fake_news_body"]],
    on="url",
    how="inner"
).reset_index(drop=True)

orig_text = (df["article_title"].fillna("") + "\n" + df["article_text"].fillna("")).str.strip()
true_text = (df["true_news_title"].fillna("") + "\n" + df["true_news_body"].fillna("")).str.strip()
fake_text = (df["fake_news_title"].fillna("") + "\n" + df["fake_news_body"].fillna("")).str.strip()

true_df = pd.DataFrame({
    "url": df["url"],
    "article_text": orig_text,
    "post_text": true_text,
    "label": 0
})
fake_df = pd.DataFrame({
    "url": df["url"],
    "article_text": orig_text,
    "post_text": fake_text,
    "label": 1
})
data = pd.concat([true_df, fake_df], ignore_index=True)

data = data[(data["post_text"].str.len() >= 200) & (data["article_text"].str.len() >= 200)].reset_index(drop=True)

print("Samples:", len(data))
print("Positives:", int((data['label']==1).sum()), "Negatives:", int((data['label']==0).sum()))

corpus_article = data["article_text"].tolist()
corpus_post    = data["post_text"].tolist()
corpus_all     = corpus_article + corpus_post

tfv = TfidfVectorizer(
    dtype=np.float32,
    sublinear_tf=True,
    strip_accents="unicode",
    analyzer="word",
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95
)
tfv.fit(corpus_all)

X_art  = tfv.transform(corpus_article)   
X_post = tfv.transform(corpus_post)

def diag_cosine_csr(A, B, eps=1e-12):
    # cos_i = <a_i, b_i> / (||a_i|| * ||b_i||)
    dots = np.array((A.multiply(B)).sum(axis=1)).ravel()
    An = np.sqrt(np.array(A.power(2).sum(axis=1)).ravel()) + eps
    Bn = np.sqrt(np.array(B.power(2).sum(axis=1)).ravel()) + eps
    return (dots / (An * Bn)).astype(np.float32).reshape(-1, 1)

cos_col = diag_cosine_csr(X_art, X_post)

X = hstack([X_post, csr_matrix(cos_col)], format="csr", dtype=np.float32)

y = data["label"].values.astype(np.int32)
groups = data["url"].values

print("Feature shape:", X.shape, "nnz:", X.nnz)
_ = check_array(X[:50].toarray(), accept_sparse=False)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(data), dtype=np.float32)
fold_metrics = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y, groups=groups), 1):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    clf = lgb.LGBMClassifier(
        objective="binary",
        learning_rate=0.05,
        n_estimators=4000,
        num_leaves=63,
        max_depth=-1,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_samples=10,  
        min_data_in_bin=1,     
        max_bin=255,
        reg_lambda=0.0,
        n_jobs=-1,
    )

    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False),]
    )

    va_pred = clf.predict_proba(X_va)[:, 1]
    oof_pred[va_idx] = va_pred
    auc = roc_auc_score(y_va, va_pred)
    fold_metrics.append(auc)
    print(f"Fold {fold} AUC: {auc:.4f}")


print("CV AUCs:", [f"{m:.4f}" for m in fold_metrics])
print("Mean AUC:", np.mean(fold_metrics).round(4))

pred_label = (oof_pred >= 0.5).astype(int)
print(confusion_matrix(y, pred_label))
print(classification_report(y, pred_label, digits=4))

best_iter = getattr(clf, "best_iteration_", None)
final_estimators = int(best_iter) if best_iter and best_iter > 0 else 1000

final_clf = lgb.LGBMClassifier(
    objective="binary",
    learning_rate=0.05,
    n_estimators=final_estimators,
    num_leaves=63,
    max_depth=-1,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_samples=10,
    min_data_in_bin=1,
    max_bin=255,
    reg_lambda=0.0,
    n_jobs=-1,
    random_state=42
)
final_clf.fit(X, y)

def score_post_with_article(article_title, article_text, post_title, post_body):
    atext = (str(article_title) + "\n" + str(article_text)).strip()
    ptext = (str(post_title) + "\n" + str(post_body)).strip()
    Xa = tfv.transform([atext])
    Xp = tfv.transform([ptext])
    dots = np.array((Xa.multiply(Xp)).sum(axis=1)).ravel()
    An = np.sqrt(np.array(Xa.power(2).sum(axis=1)).ravel()) + 1e-12
    Bn = np.sqrt(np.array(Xp.power(2).sum(axis=1)).ravel()) + 1e-12
    cos = (dots / (An * Bn)).astype(np.float32).reshape(-1, 1)
    Xf = hstack([Xp, csr_matrix(cos)], format="csr", dtype=np.float32)
    return float(final_clf.predict_proba(Xf)[:,1][0])

print("Done.")

In [ ]:
#SAMPLE TEST CASES
article_title_1 = "USCIS announces modest H‑1B filing fee adjustments for FY 2026"
article_text_1  = (
    "The U.S. immigration agency confirmed incremental changes to H‑1B petition fees for the upcoming fiscal year, "
    "noting increases in processing costs but stating that total fees remain well below prior proposals. "
    "Officials emphasized that adjustments aim to cover operational expenses while minimizing impact on employers."
)
true_title_1 = "USCIS confirms limited H‑1B fee increases for FY 2026"
true_body_1  = (
    "In a notice for the next fiscal cycle, USCIS said H‑1B petition fees will rise modestly to reflect higher processing costs. "
    "The agency said overall charges remain far below earlier drafts and are intended to sustain operations without sharply burdening sponsors."
)
fake_title_1 = "Trump hikes H‑1B visa fees to 200,000 USD per application"
fake_body_1  = (
    "The administration abruptly raised the H‑1B application fee to $200k, citing revenue needs. "
    "Companies must now pay the full amount per petition, effective immediately, according to officials."
)

article_title_2 = "September CPI shows cooling inflation as energy prices stabilize"
article_text_2  = (
    "The latest consumer price index indicates a modest monthly rise as gasoline costs eased and core services slowed. "
    "Analysts noted that shelter inflation remains sticky but overall pressures continue to moderate, "
    "keeping expectations for a gradual policy path."
)
true_title_2 = "September CPI rises slightly amid cooler energy, sticky shelter"
true_body_2  = (
    "The CPI increased modestly last month as gas prices stabilized and core services cooled. "
    "Shelter costs continued to rise at a slower pace, suggesting inflation pressures are easing overall."
)
fake_title_2 = "CPI doubles in a month as energy surges"
fake_body_2  = (
    "Prices doubled in September, driven by an extreme jump in fuel costs. "
    "Economists now warn of immediate hyperinflation unless rates are slashed."
)

print("Pair 1 (H-1B):")
p_true_1 = score_post_with_article(article_title_1, article_text_1, true_title_1, true_body_1)
p_fake_1 = score_post_with_article(article_title_1, article_text_1, fake_title_1, fake_body_1)
print(f"P(fake | TRUE post) = {p_true_1:.4f}  ")
print(f"P(fake | FAKE post) = {p_fake_1:.4f}  ")
print("OK" if p_fake_1 > p_true_1 else "Check features/params")

print("\nPair 2 (CPI):")
p_true_2 = score_post_with_article(article_title_2, article_text_2, true_title_2, true_body_2)
p_fake_2 = score_post_with_article(article_title_2, article_text_2, fake_title_2, fake_body_2)
print(f"P(fake | TRUE post) = {p_true_2:.4f}   ")
print(f"P(fake | FAKE post) = {p_fake_2:.4f}   ")
print("OK" if p_fake_2 > p_true_2 else "Check features/params")
